# 02 -- Profit giveback analysis

Paired script: `analysis/analyse_giveback.py`. Offline simulation of `ExitManager.mqh`'s
(TASK-030) two giveback-guard models against historical bar-close R paths -- **never
controls or wires to any live trading action.** Gathers the "Phase 8 evidence"
`ExitManager.mqh`'s own header says is required before either model is enabled.

**Uses clearly-labelled SYNTHETIC bar data.** Real-data run: PENDING (see final cell).

In [ ]:
import sys
import tempfile
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from analysis.analyse_giveback import run

In [ ]:
# Same R-path fixture hand-verified in tests/test_analyse_giveback.py:
# R = (close-100)/2 for a long risking 2 points -> 0.5, 1.0, 2.0, 0.7, 0.3.
# The V637 guard arms at peak 2.0R and triggers at bar 3 (R=0.7); the trade's
# actual last bar (R=0.3) is worse -- the guard would have HELPED here.
tmp_dir = Path(tempfile.mkdtemp(prefix="themba_giveback_demo_"))

pd.DataFrame(
    {
        "symbol": ["XAUUSD"] * 5,
        "timestamp": pd.date_range("2026-07-21T00:00:00Z", periods=5, freq="h"),
        "close": [101.0, 102.0, 104.0, 101.4, 100.6],
    }
).to_csv(tmp_dir / "bars.csv", index=False)

pd.DataFrame(
    [
        {
            "trade_id": "t1",
            "symbol": "XAUUSD",
            "is_long": "True",
            "entry_time": "2026-07-21T00:00:00Z",
            "exit_time": "2026-07-21T04:00:00Z",
            "entry_price": 100.0,
            "stop_price": 98.0,
        }
    ]
).to_csv(tmp_dir / "trades.csv", index=False)

In [ ]:
result = run(
    tmp_dir / "trades.csv",
    tmp_dir / "bars.csv",
    output_csv=tmp_dir / "giveback.csv",
    summary_json=tmp_dir / "summary.json",
    # Required (Codex review finding, 2026-07-22, fifth round): the bars
    # above are hourly (freq="h"), with no gaps -- see
    # trade_math.IncompleteBarCoverageError's own docstring for what this
    # now catches (a sparse bar subset silently accepted as complete
    # evidence merely because both endpoints were aligned).
    expected_cadence_minutes=60.0,
    repo_path=PROJECT_ROOT.parents[1],
)
c = result.comparisons[0]
print(f"actual_final_r    = {c.actual_final_r:.4f}")
print(f"v637_trigger_r    = {c.v637_trigger_r}")
print(f"v637_r_diff        = {c.v637_r_diff:.4f}  (positive = guard would have helped)")

assert abs(c.actual_final_r - 0.3) < 1e-9
assert abs(c.v637_trigger_r - 0.7) < 1e-9
assert abs(c.v637_r_diff - 0.4) < 1e-9

## Real-data run: PENDING

Requires a real trade+bar export -- none exists yet in this project.